In [8]:
class ReedSolomon:
    """
    Самостоятельная реализация Reed–Solomon над GF(2^8)
    сложение - XOR
    умножение - a*b = p^{(log(a)+log(b)) mod 255}
    systematic RS(n, k) с количеством проверочных символов = ecc_symbols
    """

    ############################
    # ---- GF(256) algebra ----
    ############################

    # PRIMITIVE = 285
    PRIMITIVE = 0x11D

    def __init__(self, ecc_symbols: int):
        self.ecc_symbols = ecc_symbols

        self.gf_exp = [0] * 512
        self.gf_log = [0] * 256
        self._init_tables()
        # setup tables for working GF operations

        self.gen = self._generator_poly(ecc_symbols)

    def _init_tables(self):
        x = 1
        for i in range(255):
            self.gf_exp[i] = x
            self.gf_log[x] = i
            x <<= 1
            if x & 0x100:
                x = self.gf_add(x, self.PRIMITIVE)

        for i in range(255, 512):
            self.gf_exp[i] = self.gf_exp[i - 255]

    def gf_add(self, x, y):
        return x ^ y

    def gf_mul(self, x, y):
        if x == 0 or y == 0:
            return 0
        return self.gf_exp[self.gf_log[x] + self.gf_log[y]]

    def gf_div(self, x, y):
        if x == 0:
            return 0
        if y == 0:
            raise ZeroDivisionError
        return self.gf_exp[(self.gf_log[x] - self.gf_log[y]) % 255]

    def gf_pow(self, x, p):
        return self.gf_exp[(self.gf_log[x] * p) % 255]

    def gf_inv(self, x):
        return self.gf_exp[255 - self.gf_log[x]]

    ############################
    # ---- Polynomial ops ----
    ############################

    def poly_scale(self, p, x):
        return [self.gf_mul(c, x) for c in p]

    def poly_add(self, p, q):
        # i + len(r) - len(p) - for reverse cycle
        r = [0] * max(len(p), len(q))
        for i in range(len(p)):
            r[i + len(r) - len(p)] = self.gf_add(r[i + len(r) - len(p)], p[i])
        for i in range(len(q)):
            r[i + len(r) - len(q)] = self.gf_add(r[i + len(r) - len(q)], q[i])
        return r

    def poly_mul(self, p, q):
        r = [0] * (len(p) + len(q) - 1)
        for j, b in enumerate(q):
            for i, a in enumerate(p):
                r[i + j] = self.gf_add(r[i + j], self.gf_mul(a, b))
        return r

    def poly_eval(self, p, x):
        # Horner’s method
        # P(x) = a_0*x^n + a_1*x^{n-1} + ...
        # ->
        # P(x) = (((a_0*x+a_1)*x+a_2)x+...)
        y = 0
        for c in p:
            y = self.gf_add(self.gf_mul(y, x), c)
        return y

    ############################
    # ---- Generator ----
    ############################

    def _generator_poly(self, nsym):
        g = [1]
        for i in range(nsym):
            g = self.poly_mul(g, [1, self.gf_pow(2, i)])
        # g(x) = (x-2^0)(x-2^1)(x-2^2)...(x-2^{nsym-1})
        return g

    ############################
    # ---- Encode ----
    ############################

    def encode(self, data: str, verbose: bool = False) -> list[int]:
        data_list = [ord(i) for i in data]
        if r := list(filter(lambda x: x >= 2**8, data_list)):
            raise ValueError(f"символы с большими кодами: {r}")

        msg = data_list + [0] * self.ecc_symbols

        for i in range(len(data_list)):
            coef = msg[i]
            if coef != 0:
                for j, g in enumerate(self.gen):
                    msg[i + j] = self.gf_add(msg[i + j], self.gf_mul(g, coef))

        encoded = data_list + msg[-self.ecc_symbols:]

        # msg = self.poly_mul(data_bytes, self.gen)
        # encoded = data_bytes + msg

        if verbose:
            print(data, "- исходный текст")
            print(data_list, "- в байтах")
            print(encoded, "- код Рида-Соломона")

        return encoded

    ############################
    # ---- Syndrome ----
    ############################

    def _calc_syndromes(self, msg):
        return [0] + [self.poly_eval(msg, self.gf_pow(2, i)) for i in range(self.ecc_symbols)]

    ############################
    # ---- Berlekamp–Massey ----
    ############################

    def _berlekamp_massey(self, synd):
        err_loc = [1]
        old = [1]

        for i in range(1, len(synd)):
            delta = synd[i]
            for j in range(1, len(err_loc)):
                delta = self.gf_add(delta, self.gf_mul(err_loc[-(j + 1)], synd[i - j]))

            old.append(0)

            if delta != 0:
                if len(old) > len(err_loc):
                    new = self.poly_scale(old, delta)
                    old = self.poly_scale(err_loc, self.gf_inv(delta))
                    err_loc = new

                err_loc = self.poly_add(err_loc, self.poly_scale(old, delta))

        return err_loc

    ############################
    # ---- Chien search ----
    ############################

    def _find_errors(self, err_loc, nmess):
        errs = len(err_loc) - 1
        err_pos = []

        for i in range(nmess):
            if self.poly_eval(err_loc, self.gf_pow(2, 255 - i)) == 0:
                err_pos.append(nmess - 1 - i)

        if len(err_pos) != errs:
            raise ValueError("Too many errors to correct")

        return err_pos

    ############################
    # ---- Forney ----
    ############################

    def _forney(self, msg, err_pos):

        for pos in err_pos:
            msg[pos] = self.gf_add(msg[pos], 0xFF)

        return msg


    ############################
    # ---- Fix errors ----
    ############################

    def fix_errors(self, encoded_data: list[int], verbose: bool = False) -> tuple[list[int], int]:
        msg = list(encoded_data)

        synd = self._calc_syndromes(msg)

        if max(synd) == 0:
            if verbose:
                print("ошибок нет")
            return encoded_data, 0

        err_loc = self._berlekamp_massey(synd)
        err_pos = self._find_errors(err_loc, len(msg))
        if verbose:
            print(err_pos, "- на этих местах найдены ошибки")

        corrected = self._forney(msg, err_pos)

        if verbose:
            print(encoded_data, "- до исправления")
            print(corrected, "- после исправления")

        return corrected, len(err_pos)

    ############################
    # ---- Decode ----
    ############################

    def decode(self, encoded_data: list[int], verbose: bool = False) -> str:
        data = list(encoded_data)
        data = data[:-self.ecc_symbols]
        decoded_str = "".join([chr(i) for i in data])

        if verbose:
            print(encoded_data, "- правильный код Рида-Соломона")
            print(data)
            print(decoded_str)

        return decoded_str

    ############################
    # ---- Error simulation ----
    ############################

    def simulate_errors(self, encoded_data: bytearray, error_positions: list[int], verbose: bool = False) -> list[int]:
        corrupted = list(encoded_data)

        for pos in error_positions:
            if pos < len(corrupted):
                corrupted[pos] = self.gf_add(corrupted[pos], 0xFF)

        if verbose:
            print(encoded_data, "- код Рида-Соломона")
            print(corrupted, "- поломанный код")

        return corrupted


In [9]:
rs = ReedSolomon(ecc_symbols=4)  # ecc_symbols//2 ошибок

In [10]:
original_text = "hello"

In [ ]:
encoded = rs.encode(original_text, verbose=True)

hello - исходный текст
[104, 101, 108, 108, 111] - в байтах
[104, 101, 108, 108, 111, 203, 186, 169, 186] - код Рида-Соломона


In [12]:
encoded = rs.simulate_errors(encoded, error_positions=[0,2], verbose=True)

[104, 101, 108, 108, 111, 203, 186, 169, 186] - код Рида-Соломона
[151, 101, 147, 108, 111, 203, 186, 169, 186] - поломанный код


In [13]:
encoded, errors_fixed = rs.fix_errors(encoded, verbose=True)

[2, 0] - на этих местах найдены ошибки
[151, 101, 147, 108, 111, 203, 186, 169, 186] - до исправления
[104, 101, 108, 108, 111, 203, 186, 169, 186] - после исправления


In [14]:
decoded = rs.decode(encoded, verbose=True)

[104, 101, 108, 108, 111, 203, 186, 169, 186] - правильный код Рида-Соломона
[104, 101, 108, 108, 111]
hello
